In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import statsmodels.stats.proportion as CP


# Load data

In [ ]:
Genta = pd.read_csv(f'Tables/Gentamicin_measured.csv',index_col=0)
Genta_threshold=[20,20,40,40]

Tetra = pd.read_csv(f'Tables/Tetracycline_measured.csv',index_col=0)
Tetra_threshold=[20,20,20,20]


Chp = pd.read_csv(f'Tables/Chloramphenicol_measured.csv',index_col=0)
Chp_threshold=[40,20,30,20,20,20,30,20,30,30,40,20,40,20]

Cipro = pd.read_csv(f'Tables/Ciprofloxacin_measured.csv',index_col=0)
Cipro_threshold=[20,20,20,20]

In [ ]:
Genta

In [ ]:
def AnalyseNDependence(data,threshold,alpha):
    
    data=data[data['n_cells']<threshold]
    data['dead']=data.n_cells_final < threshold
    data['dead'] =  data.dead.replace({True: 1, False: 0})

    out = data.groupby(['n_cells']).dead.mean().reset_index()
    out = out.rename(columns={'dead': 'prob_neg'})

    CountData_detected = data.groupby(['n_cells']).dead.count().reset_index()
    out['count'] =CountData_detected['dead'].values
    out['num_neg'] =out['count'].values * out['prob_neg'].values 

    cp=CP.proportion_confint(out['num_neg'].values  , out['count'].values, alpha=alpha, method='beta')
    #with respect to estimated value so plotting is easier
    out['lower'] = out['prob_neg'].values - cp[0]
    out['upper'] = cp[1]-out['prob_neg'].values
  
    return out




In [ ]:
alpha=0.05

Genta_n_dependence = (
    Genta[(Genta.dataset==0)]
    .groupby('concentration')
    .apply(lambda x: AnalyseNDependence(x,20,alpha)) # match threshholds
    .droplevel(level=1)
    .reset_index()

)


Tetra_n_dependence = (
    Tetra[(Tetra.dataset==0)]
    .groupby('concentration')
    .apply(lambda x: AnalyseNDependence(x,20,alpha)) # match threshholds
    .droplevel(level=1)
    .reset_index()

)


Cipro_n_dependence = (
    Cipro[(Cipro.dataset==0)]
    .groupby('concentration')
    .apply(lambda x: AnalyseNDependence(x,20,alpha)) # match threshholds
    .droplevel(level=1)
    .reset_index()

)


Chp_n_dependence = (
    Chp[(Chp.dataset==5)]
    .groupby('concentration')
    .apply(lambda x: AnalyseNDependence(x,20,alpha)) # match threshholds
    .droplevel(level=1)
    .reset_index()

)

# Plot 

In [ ]:
def plot_n_dependence(data,ax,leg_title,line_width):

    ax.set_yscale('log',base=2)
    colors=sns.color_palette("colorblind")

    for i,c in enumerate(data.concentration.unique()):
        d = data[data['concentration'] == c]

        ax.errorbar(
            d['n_cells'],
            d['prob_neg'],
            yerr=[d['lower'], d['upper']],
            marker='o',
            linestyle='-',
            color=colors[i],
            label=f"{c}"
        )
     
    ax.legend(
        title=leg_title,
        loc="center left",
        bbox_to_anchor=(1.02, 0.8),
        frameon=True,
        borderaxespad=0
    )
    


In [ ]:
# Define once, use everywhere
standard_linewidth = 1.5  # Choose a value that looks good

spinesParams = {
    'axes.spines.right': True,
    'axes.spines.top': True,
    'axes.linewidth': standard_linewidth,
}



tex_fonts = {
    # Use LaTeX to write all text
    "text.usetex": True,
    "font.family": "serif",
    # Use 10pt font in plots, to match 10pt font in document
    "axes.labelsize": 10,
    "font.size": 10,
    # Make the legend/label fonts a little smaller
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
}
tickParams = {
    "xtick.top":True,
    "xtick.bottom":True,
    "xtick.direction": "in",
    "ytick.left":True,
    "ytick.right":True,
    "ytick.direction": "in",
}
spinesParams = {
    'axes.spines.right' : True,
    'axes.spines.top' : True
}
plt.rcParams.update(tex_fonts)
plt.rcParams.update(tickParams)
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}\usepackage{amssymb}'
plt.rcParams.update(spinesParams)  # ADD THIS LINE

inchPerCm = 0.393701
goldenRatio = (1+np.sqrt(5))/2
figWidthCm = 17.9
#figHeightCm = figWidthCm
figHeightCm = figWidthCm*1.2

figWidthInches = figWidthCm*inchPerCm
figHeightInches = figHeightCm*inchPerCm



lw=1
marker_size=10
lw2=1

fig_label_x=-0.15
fig_label_y=1.05

In [ ]:
xmax=[5,5,5,5]
xmin=[-0.2,-0.2,-0.2,-0.2]

In [ ]:
marker_size=5
colors=sns.color_palette("colorblind")

fig_label_x_1=-0.17
fig_label_y_1=1

fig, ax = plt.subplots(2,2, figsize=(figWidthInches, figHeightInches))


plot_n_dependence(Genta_n_dependence,ax[0,0],r'$\left[\frac{\mu g}{ml}\right]$',1)
plot_n_dependence(Cipro_n_dependence,ax[0,1],r'$\left[\frac{n g}{ml}\right]$',1)
plot_n_dependence(Chp_n_dependence,ax[1,0],r'$\left[\frac{\mu g}{ml}\right]$',1)
plot_n_dependence(Tetra_n_dependence,ax[1,1],r'$\left[\frac{\mu g}{ml}\right]$',1)


ax[0,0].set_ylabel('')
ax[0,0].set_title(r'Gentamicin')


ax[0,1].set_ylabel('')
ax[0,1].set_title(r'Ciprofloxacin')

ax[1,0].set_ylabel(r'$q_{|\tilde{n}}$',rotation=0)
ax[1,0].yaxis.set_label_coords(-0.15, 1.05)  # Position at middle of ax[0,0] and ax[1,0]

ax[1,0].set_title(r'Tetracycline')

                  
ax[1,1].set_ylabel('')
ax[1,1].set_title(r'Chloramphenicol')
ax[1,0].set_xlabel(r'$\tilde{n}$')
ax[1,0].xaxis.set_label_coords(1.1, -0.05)  # Position at middle of ax[1,0] and ax[1,1]



ax[0,0].set_xlim([xmin[0],xmax[0]])
ax[0,0].set_ylim([0,1.5])

ax[0,1].set_xlim([xmin[1],xmax[1]])
ax[0,1].set_ylim([0,1.5])

ax[1,0].set_xlim([xmin[2],xmax[2]])
ax[1,0].set_ylim([0,1.5])

ax[1,1].set_xlim([xmin[3],xmax[3]])
ax[1,1].set_ylim([0,1.5])



# Adjust spacing - hspace controls vertical spacing between rows
plt.subplots_adjust(hspace=0.2,wspace=0.4)

plt.show()

In [ ]:
baseSavePath=''
saveFigPath = os.path.join(baseSavePath,'Figure_n_dep.pdf')
fig.savefig(saveFigPath,bbox_inches='tight')